In [1]:
from dataclasses import dataclass

@dataclass
class ModelParameters:
    vocab_size: int  # total token in vocabulary
    d_model:int   # embedding dimension
    max_seq_len:int  # context window 
    num_layers: int # here layers mean decoder block
    num_heads: int # no. of attention head

In [2]:
model_params = ModelParameters(
    vocab_size=44,
    d_model = 32,
    max_seq_len=5,
    num_layers=1,
    num_heads=4
)

In [3]:
from torch import nn
import torch

In [7]:
class EmbeddingLayer(nn.Module):
    def __init__(self,model_params:ModelParameters):
        super().__init__()

        """
        Always remember that embedding layer is not actually a neural laer with any activation function ,
        its just an 2d matrix lookup the matrix size is (vocab_size,d_model) but but but the d_model here is lets one array
         so that array is already have learnable parameter means in our case that 32 dim arrays have the values from the training 
         for e.g:
         token_embeding[1] will be the embeding of token id 1 which is 32 dim embedding array
        """
        self.token_embeddings=nn.Embedding(model_params.vocab_size,model_params.d_model)
        self.pos_embeddings = nn.Embedding(model_params.max_seq_len,model_params.d_model)

    def forward(self,token_ids):
        pos_ids = torch.arange(token_ids.shape[1])
        embedding = self.token_embeddings(token_ids) + self.pos_embeddings(pos_ids)
        return embedding


In [8]:
embedding_layer = EmbeddingLayer(model_params)

In [11]:
x = torch.randint(0,model_params.vocab_size,(2,5))
x

tensor([[40,  7, 27,  0,  5],
        [12, 16, 12, 18, 13]])

In [13]:
x_embed = embedding_layer(x)
x_embed.shape

torch.Size([2, 5, 32])

In [20]:
w_q = nn.Linear(model_params.d_model,model_params.d_model,bias=False)
w_k = nn.Linear(model_params.d_model,model_params.d_model,bias=False)
w_v = nn.Linear(model_params.d_model,model_params.d_model,bias=False)

In [19]:
w_q

Linear(in_features=32, out_features=32, bias=False)

In [30]:
q = w_q(x_embed)
k = w_k(x_embed)
v = w_v(x_embed)

In [31]:
q.shape

torch.Size([2, 5, 32])

In [32]:
x.shape

torch.Size([2, 5])

In [35]:
# here 32 is d_model and this for single head but for multihead we need same of (b,t,num_head,head_dim) here head_dim = d_model//num_heads
# head_dim = 32//4 = 8
head_dim = model_params.d_model//model_params.num_heads
q = torch.reshape(q,(x.shape[0],x.shape[1],model_params.num_heads,head_dim))
k = torch.reshape(k,(x.shape[0],x.shape[1],model_params.num_heads,head_dim))
v = torch.reshape(v,(x.shape[0],x.shape[1],model_params.num_heads,head_dim))

q.shape

torch.Size([2, 5, 4, 8])

In [36]:
 # so that pytorch can treat all num_head parallely we need to swap middle dimension in q,k and v ,
 # means the final shape will be (b,num_head,t,head_dim)
q = q.transpose(1,2)
k = k.transpose(1,2)
v = v.transpose(1,2)
q.shape

torch.Size([2, 4, 5, 8])

In [38]:
scores = q @ k.transpose(-2,-1) / (head_dim ** 0.5)
scores.shape

torch.Size([2, 4, 5, 5])

In [43]:
scores[0][0][0] 
# relation of one words to each words of that sentence in the
#  first head of the first batch

tensor([-0.5578,  0.5924,  0.6641, -0.4652,  0.3069],
       grad_fn=<SelectBackward0>)

In [48]:
casual_mask = torch.tril(torch.ones(scores.shape[-2],scores.shape[-1]))
masked_score = scores.masked_fill(casual_mask==0,float("-inf"))
masked_score.shape


torch.Size([2, 4, 5, 5])

In [ ]:
attention_weights = (torch.softmax(masked_score,dim = -1)) 
attention_weights.shape

torch.Size([2, 4, 5, 5])

In [60]:
context_vector = attention_weights@v
context_vector.shape

torch.Size([2, 4, 5, 8])

In [63]:
# now first we need to swap back to the original size which was b,t,num_head,head_dim

context_vector = context_vector.transpose(1,2)

In [64]:
context_vector.shape

torch.Size([2, 5, 4, 8])

In [67]:
context_vector = context_vector.contiguous().view(x_embed.shape)

In [68]:
w_o = nn.Linear(model_params.d_model,model_params.d_model)
final_output = w_o(context_vector)
final_output.shape

torch.Size([2, 5, 32])

In [71]:
class MultiHeadAttention(nn.Module):
    def __init__(self,model_params:ModelParameters):
        super().__init__()

        self.config = model_params
        # initialize q,k,v
        self.w_q = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_k = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_v = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

        self.register_buffer(
                            "casual_mask", 
                            torch.tril(
                                torch.ones(
                                    self.config.max_seq_len, 
                                    self.config.max_seq_len
                                    )
                                )
                        )
        
        self.head_dim = self.config.d_model//self.config.num_heads
        

        self.w_o = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

    def forward(self,x):
        # here x is the embeding and not raw x  and b is batch size, t is the current sequence length and d_model is hidden dimension
        # shape of x -> (b,t,d_model)
        q = self.w_q(x) # (b,t,d_model)
        k = self.w_k(x) # (b,t,d_model)
        v = self.w_v(x) # (b,t,d_model)

        q = torch.reshape(q,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        k = torch.reshape(k,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        v = torch.reshape(v,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)


        q = q.transpose(1,2) # (b,num_head,t,head_dim)
        k = k.transpose(1,2) # (b,num_head,t,head_dim)
        v = v.transpose(1,2) # (b,num_head,t,head_dim)

        scores = q @ k.transpose(-2,-1) / (self.head_dim ** 0.5) # (b,num_head,t,t)
        mask = self.casual_mask[:x.shape[1],:x.shape[1]] # (t,t)
        masked_score = scores.masked_fill(mask==0,float("-inf")) # (b,num_head,t,t)
        attention_weights = torch.softmax(masked_score,dim = -1) # (b,num_head,t,t)
        context_vector = attention_weights @ v # (b,num_head,t,head_dim)

        context_vector = context_vector.transpose(1,2) # (b,t,num_head,head_dim) 
        context_vector = context_vector.contiguous().view(x.shape) # (b,t,d_model)
        return self.w_o(context_vector) #(b,t,d_model) which is same as the input embedding 


In [72]:
multi_head_attention_layer  = MultiHeadAttention(model_params)

In [73]:
output = multi_head_attention_layer(x_embed)
output.shape

torch.Size([2, 5, 32])